# Telegram AI Video Bot — Kaggle Worker
Attach your uploaded dataset (see cell 1), enable **GPU P100** in the accelerator panel.

### Setup
1. Go to *Datasets → New Dataset*, upload the `telegram-ai-video-bot` folder from your PC.
2. Click *Add Data* on this notebook and attach that dataset.
3. Run cells below. No internet needed for code — pip packages will install once.

In [ ]:
import os, shutil, subprocess, sys

BASE = '/kaggle/working/telegram-ai-video-bot'
if os.path.exists(BASE):
    shutil.rmtree(BASE, ignore_errors=True)

# Find uploaded dataset (adjust name if yours differs)
DATASET_DIR = '/kaggle/input'
entries = os.listdir(DATASET_DIR)
print("Available datasets:", entries)
src = os.path.join(DATASET_DIR, entries[0]) if entries else None

if src and os.path.isdir(src):
    shutil.copytree(src, BASE)
    print(f"Copied {src} -> {BASE}")
else:
    raise FileNotFoundError(f"No dataset found in {DATASET_DIR}. Upload your repo as a dataset first.")

%cd $BASE

# Install deps (runs once; cached by Kaggle)
!pip install -r worker/requirements.txt -q

In [ ]:
!touch shared/__init__.py worker/__init__.py worker/models/__init__.py dispatcher/__init__.py scripts/__init__.py
print("Created __init__.py files")

In [ ]:
from kaggle_secrets import UserSecretsClient

client = UserSecretsClient()
os.environ["SUPABASE_URL"] = client.get_secret("SUPABASE_URL")
os.environ["SUPABASE_KEY"] = client.get_secret("SUPABASE_KEY")
os.environ["HF_TOKEN"] = client.get_secret("HF_TOKEN")
os.environ["GROQ_API_KEY"] = client.get_secret("GROQ_API_KEY")

models_dir = "/kaggle/working/models"
os.makedirs(models_dir, exist_ok=True)
os.environ["MODELS_DIR"] = models_dir
print("Secrets loaded. Models dir:", models_dir)

In [ ]:
!python scripts/download_models.py ltx-video

In [ ]:
!python -m worker.gpu_detect

!python -m worker.worker --provider kaggle --model auto